In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT))




import pandas as pd
import numpy as np
from matplotlib import pyplot as plt
import seaborn as sns


from ETL.ETL_PIPELINE import  ETL


In [4]:


path = Path.cwd().parent.parent  /"DATASET"/"water_potability.csv"

df = ETL.run(path)

df.head()





[OK] Column validation passed.
[OK] Data type validation passed.
[OK] Dataset validation successful.
                 DATASET INSPECTION REPORT

GENERAL INFORMATION
----------------------------------------------------------------------
Dimensions             : (3276, 10)
Rows                   : 3,276
Columns                : 10
Duplicate Rows         : 0
Duplicate Rows (%)     : 0.00%

COLUMN INFORMATION
----------------------------------------------------------------------
                Data Type  Non-Missing  Missing  Missing %  Duplicate Values %  Duplicated Column
chloramines       float64         3276        0       0.00                0.00              False
conductivity      float64         3276        0       0.00                0.00              False
hardness          float64         3276        0       0.00                0.00              False
organic_carbon    float64         3276        0       0.00                0.00              False
ph                float64     

,ph,hardness,solids,chloramines,sulfate,conductivity,organic_carbon,trihalomethanes,turbidity,potability
0,NaN,204.890455,20791.318981,7.300212,368.516441,564.308654,10.379783,86.990970,2.963135,0
1,3.716080,129.422921,18630.057858,6.635246,NaN,592.885359,15.180013,56.329076,4.500656,0
2,8.099124,224.236259,19909.541732,9.275884,NaN,418.606213,16.868637,66.420093,3.055934,0
3,8.316766,214.373394,22018.417441,8.059332,356.886136,363.266516,18.436524,100.341674,4.628771,0
4,9.092223,181.101509,17978.986339,6.546600,310.135738,398.410813,11.558279,31.997993,4.075075,0


In [7]:
print(df.isna().sum())

ph                 491
hardness             0
solids               0
chloramines          0
sulfate            781
conductivity         0
organic_carbon       0
trihalomethanes    162
turbidity            0
potability           0
dtype: int64


In [5]:
df_imputed = df.fillna(df.median(numeric_only=True))

print(df_imputed.isna().sum())

ph                 0
hardness           0
solids             0
chloramines        0
sulfate            0
conductivity       0
organic_carbon     0
trihalomethanes    0
turbidity          0
potability         0
dtype: int64


In [13]:
import numpy as np

data = df_imputed.drop(columns=["potability"])
data.head()

,ph,hardness,solids,chloramines,sulfate,conductivity,organic_carbon,trihalomethanes,turbidity
0,7.036752,204.890455,20791.318981,7.300212,368.516441,564.308654,10.379783,86.990970,2.963135
1,3.716080,129.422921,18630.057858,6.635246,333.073546,592.885359,15.180013,56.329076,4.500656
2,8.099124,224.236259,19909.541732,9.275884,333.073546,418.606213,16.868637,66.420093,3.055934
3,8.316766,214.373394,22018.417441,8.059332,356.886136,363.266516,18.436524,100.341674,4.628771
4,9.092223,181.101509,17978.986339,6.546600,310.135738,398.410813,11.558279,31.997993,4.075075


In [14]:
from statsmodels.stats.outliers_influence import variance_inflation_factor

# Select only numerical predictor variables
X = data
# Calculate VIF
vif = pd.DataFrame()

vif["Feature"] = X.columns

vif["VIF"] = [
    variance_inflation_factor(X.values, i)
    for i in range(X.shape[1])
]

print(vif)

           Feature       VIF
0               ph  1.016536
1         hardness  1.019982
2           solids  1.039030
3      chloramines  1.008197
4          sulfate  1.035744
5     conductivity  1.002211
6   organic_carbon  1.004029
7  trihalomethanes  1.001979
8        turbidity  1.002973


The VIF values for all predictor variables are close to 1, ranging from approximately 1.002 to 1.039. This indicates that there is no significant multicollinearity among the predictors.

So based specifically on multicollinearity, there is no reason to remove any of these features.

In [6]:
from sklearn.feature_selection import mutual_info_classif
from PROCESSING.DATA_PROCESSING import DataProcessor



processor = DataProcessor(df_imputed)

X_train, X_test, y_train, y_test = processor.process()


mi_scores = mutual_info_classif(
    X_train,
    y_train,
    random_state=42
)

mi_df = pd.DataFrame({
    "Feature": X_train.columns,
    "MI": mi_scores
})

mi_df = mi_df.sort_values(
    by="MI",
    ascending=False
)

print(mi_df)



           Feature        MI
1         hardness  0.022567
0               ph  0.014219
4          sulfate  0.008019
5     conductivity  0.007439
6   organic_carbon  0.007346
8        turbidity  0.002908
2           solids  0.000000
3      chloramines  0.000000
7  trihalomethanes  0.000000
